## Tracking of a EURUSD dataset optimised over a parameter grid, using likelihoods

In [ ]:
## Non-optimised params
optimise_over_T_timesteps= 450 #If using binder, change to lower num of steps (200 confimred to work)
filter_over_T_timesteps= 450
seed = 1 # Random seem for reproducibility
c=10

In [ ]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import os
import pandas as pd
from stonesoup.types.detection import Detection
from stonesoup.types.groundtruth import GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater

# Step 1: Load the CSV file
folder = "TrackedDatasets"
file_name = "EURCHF_Ticks_03.02.2025-03.02.2025.csv"  # Example file
excel_file = os.path.join(folder, file_name)
 
data = pd.read_csv(excel_file)
tick_times = data['Local time']
filter_over_T_timesteps= min(filter_over_T_timesteps,len(tick_times))
optimise_over_T_timesteps= min(optimise_over_T_timesteps,filter_over_T_timesteps)

# Step 2: Extract relevant columns
tick_times = data['Local time'][:filter_over_T_timesteps]
mid_rates = data['Mid'][:filter_over_T_timesteps]
Ask_rates = data['Ask'][:filter_over_T_timesteps]
Bid_rates = data['Bid'][:filter_over_T_timesteps]

# Step 3: Generates Timestamps
fmt = r"%d.%m.%Y %H:%M:%S.%f GMT%z"
timesteps = pd.to_datetime(tick_times, format=fmt)
start_time=timesteps[0]


In [ ]:
## Determine prior values
number_particles = 2000
initial_rate=mid_rates[0]
# prior_mean =np.array([initial_rate,0]) 
prior_mean =np.array([initial_rate]) 


pos_std = 0.001 * initial_rate  # e.g. 0.1% of initial rate
vel_std = 0.0005               # say 0.5 pip
# prior_covar = np.diag([pos_std**2, vel_std**2])
prior_covar= np.diag([pos_std**2])

# Compute typical bid-ask spread
spreads = (Ask_rates - Bid_rates)
avg_spread = np.mean(spreads)   # or median
# measurement sigma ~ half that spread
meas_sigma = 0.5 * avg_spread
meas_sig2 = meas_sigma**2

In [ ]:
## Initiate Priors
# Sample from the prior Gaussian distribution
from stonesoup.types.array import CovarianceMatrices
from stonesoup.types.array import StateVector


# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(
    mean=prior_mean,  # Initial state: [price, dP/dt]
    cov=prior_covar,  # Covariance for the initial state
    size=number_particles
)

# Define covariance for particle array
covars =     [prior_covar for _ in range(number_particles)]

# Create prior particle state
lp_prior = MarginalisedParticleState(
    state_vector=StateVectors(np.atleast_2d(states)),  # Transpose states to shape (2, N)
    covariance=CovarianceMatrices(covars),  # Covariance matrix
    weight=np.array([Probability(1 / number_particles)] * number_particles),
    timestamp=start_time-timedelta(milliseconds=1)
)

gp_prior=GaussianState(state_vector=prior_mean,
                        covar=prior_covar,
                        timestamp=start_time-timedelta(milliseconds=1))

In [ ]:
## Build the Observed mid_rates as noisy measurements

# Define Measurement Model
measurement_model = LinearGaussian(
    ndim_state=1,  # State vector dimensions: [price, dP/dt] for the models
    mapping=[0],  # Map the measurement to the 'price' dimension
    noise_covar=np.diag([meas_sig2])  # Small measurement noise for 'price'
)

# Generates Measurements from Ground Truth
observed_mid=Track()
observed_bid=Track()
observed_ask=Track()

measurements = []
for i in range(len(timesteps)):
    timestamp = timesteps[i]
    mid= GroundTruthState(state_vector=np.atleast_2d(mid_rates[i]),timestamp=timestamp)
    bid= GroundTruthState(state_vector=np.atleast_2d(Bid_rates[i]),timestamp=timestamp)
    ask= GroundTruthState(state_vector=np.atleast_2d(Ask_rates[i]),timestamp=timestamp)

    observed_mid.append(mid)
    observed_bid.append(bid)
    observed_ask.append(ask)

    measurements.append(Detection(
        state_vector=np.atleast_1d(mid_rates[i]),
        timestamp=timestamp,
        measurement_model=measurement_model
    ))

In [ ]:
## Outlining the parameter grids over which we'll estimate
# 'shared' parameters     
noise_diff_coeffs = np.logspace(-15, -10, num=6)        
damping_coeffs = np.linspace(0.05, 0.2, 1)  

# LP parameters
a=5
b=3
pre_1_alpha= np.linspace(0.5, 0.9, a)
post_1_alpha = np.linspace(1.1,1.9,b) 
alpha_values = np.zeros(a+b)
alpha_values[:a],alpha_values[a:a+b]=pre_1_alpha,post_1_alpha

mu_values = np.linspace(-0.0001, 0.0001, 5)                        

In [ ]:
## Generates the necessary dicts for all the components to save time in the loop
# LP component dicts
from stonesoup.models.transition.levy_linear import LevyRandomWalk
from stonesoup.models.transition.linear import OrnsteinUhlenbeck


lp_predictors= {}
resampler = SystematicResampler()

# GP component dicts
gp_predictors = {}

for sigma_W2 in noise_diff_coeffs:
    for theta in damping_coeffs:
        for mu_W in mu_values:
            for alpha in alpha_values:
                        #Generates all the Levy process predictors and updaters
                        lp_driver = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, noise_case=NoiseCase(2))
                        lp_transition_model = LevyRandomWalk(driver=lp_driver,noise_diff_coeff=sigma_W2)
                        lp_predictor = MarginalisedParticlePredictor(transition_model=lp_transition_model)
                        lp_predictors[(sigma_W2,theta, mu_W,alpha)] = lp_predictor

        #Generates all the Gaussian process predictors and updaters
        gp_transition_model= RandomWalk(noise_diff_coeff=sigma_W2)
        gp_predictor = KalmanPredictor(gp_transition_model)
        gp_predictors[(sigma_W2,theta)] = gp_predictor

# Build the updaters that depend on measurement model only
lp_updater = MarginalisedParticleUpdater(measurement_model, resampler)
gp_updater = KalmanUpdater(measurement_model)        

In [ ]:
## Filtering and likelihood calculation
from scipy.special import logsumexp
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

#Track dicts
lp_tracks = {}
gp_tracks = {}

#Likelihood dicts
lp_likelihoods = {}
gp_likelihoods = {}

for i, meas in enumerate(measurements[:optimise_over_T_timesteps]):
    for sigma_W2 in noise_diff_coeffs:
        for theta in damping_coeffs:
            for mu_W in mu_values:
                for alpha in alpha_values:
                    if (sigma_W2,theta, mu_W,alpha) not in lp_likelihoods:
                        lp_tracks[(sigma_W2,theta, mu_W,alpha)]=Track()
                        prior=lp_prior
                        lp_likelihoods[(sigma_W2,theta, mu_W,alpha)] = 0.0
                    else:
                        lp_tracks[(sigma_W2,theta, mu_W,alpha)]
                        prior = lp_tracks[(sigma_W2,theta, mu_W,alpha)][-1]
                        
                    lp_predictor = lp_predictors[(sigma_W2,theta, mu_W,alpha)]
                    lp_prediction = lp_predictor.predict(prior, timestamp=meas.timestamp)
                    lp_hypothesis = SingleHypothesis(lp_prediction, meas)                    
                    lp_post = lp_updater.update(lp_hypothesis)
                    lp_tracks[(sigma_W2,theta, mu_W,alpha)].append(lp_post)
                    # Accumulate log-likelihood
                    lp_likelihoods[(sigma_W2,theta, mu_W,alpha)] += logsumexp(lp_updater.measurement_model.logpdf(meas,lp_post))-np.log(number_particles*optimise_over_T_timesteps)

            if (sigma_W2,theta) not in gp_likelihoods:
                gp_tracks[(sigma_W2,theta)]=Track()
                prior=gp_prior
                gp_likelihoods[(sigma_W2,theta)] = 0.0
            else:
                gp_tracks[(sigma_W2,theta)]
                prior = gp_tracks[(sigma_W2,theta)][-1]

            gp_predictor = gp_predictors[(sigma_W2,theta)]
            gp_prediction = gp_predictor.predict(prior, timestamp=meas.timestamp)
            gp_hypothesis = SingleHypothesis(gp_prediction, meas)
            gp_post = gp_updater.update(gp_hypothesis)
            gp_tracks[(sigma_W2,theta)].append(gp_post)
            
            gp_likelihoods[(sigma_W2,theta)] += gp_updater.measurement_model.logpdf(meas,gp_post) -np.log(optimise_over_T_timesteps)
    if (i+1)//100 == (i+1)/100:
        print(f"measurement {i+1} of {optimise_over_T_timesteps}")

In [ ]:
## Define likelihood organiser and output optimal params
def summarize_top_likelihoods(lp_likelihoods, gp_likelihoods, top_n=5):
    """
    Summarize and print the top-N parameter configurations with the highest 
    log-likelihood for both the Lévy process and Gaussian process models.
    
    Parameters
    ----------
    lp_likelihoods : dict
        Dictionary keyed by (mu_W, theta, alpha, sigma_W2, meas_sig2),
        with values = total (log) likelihood.
    gp_likelihoods : dict
        Dictionary keyed by ((sigma_W2,theta), meas_sig2), with values = total (log) likelihood.
    top_n : int, optional
        Number of highest-likelihood entries to show for each model. Default=5.
    
    Returns
    -------
    list_of_top_lp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Lévy model.
    list_of_top_gp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Gaussian process model.
    """

    # --- 1) Sort the Lévy model likelihoods ---
    #   lp_likelihoods is keyed by (mu_W, theta, alpha, sigma_W2, meas_sig2)
    #   The value is the total log-likelihood
    # We'll get items as ((sigma_W2,theta, mu_W,alpha), loglike)
    list_of_lp = list(lp_likelihoods.items())
    # Sort descending by log-likelihood
    list_of_lp.sort(key=lambda x: x[1], reverse=True)
    # Take top_n
    list_of_top_lp = list_of_lp[:top_n]

    # --- 2) Sort the Gaussian process likelihoods ---
    #   gp_likelihoods is keyed by (sigma_W2,theta)
    list_of_gp = list(gp_likelihoods.items())
    list_of_gp.sort(key=lambda x: x[1], reverse=True)
    list_of_top_gp = list_of_gp[:top_n]

    # --- 3) Print summary in a neat format ---
    print("=== Lévy Random Walk - Top {} Log-Likelihoods ===".format(top_n))
    for rank, ((sigma_W2,theta, mu_W,alpha), loglike) in enumerate(list_of_top_lp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | sigma_W2={sigma_W2}, alpha={alpha}") #, theta={theta} , mu={mu_W}")

    print("")
    print("=== Gaussian Random Walk - Top {} Log-Likelihoods ===".format(top_n))
    for rank, ((sigma_W2,theta), loglike) in enumerate(list_of_top_gp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | sigma_W2={sigma_W2}") #, theta={theta}")

    return list_of_top_lp, list_of_top_gp

In [ ]:
list_of_top_lp, list_of_top_gp = summarize_top_likelihoods(lp_likelihoods,gp_likelihoods,top_n=500)

In [ ]:
## Path to save plots in
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSDplots"

In [ ]:
## Optimal parameters used for tracks and plotting
lp_optimal_params,gp_optimal_params=list_of_top_lp[2][0],list_of_top_gp[0][0]

label="Price"
particle_plotter_dict = {}

In [ ]:
## Track (longer) sequence using optimised params

# Generate predictions A,B and C seconds into the future
prediction_horizons={
    "A":5,
    "B":30,
    "C":60,
}

LP_track= Track()
GP_track=Track()
# lp_predictor = lp_predictors[lp_optimal_params]
# gp_predictor = gp_predictors[gp_optimal_params]

# lp_prediction_tracks={}
# gp_prediction_tracks={}


for i, meas in enumerate(measurements):
    if i==0:
        prior = lp_prior
    else:
        prior = LP_track[-1]     

    lp_prediction = lp_predictor.predict(prior, timestamp=meas.timestamp)
    lp_hypothesis = SingleHypothesis(lp_prediction, meas)                    
    lp_post = lp_updater.update(lp_hypothesis)
    LP_track.append(lp_post)
    # for horizon_key in prediction_horizons:
    #     lp_forecasted_prediction = lp_predictor.predict(lp_post, timestamp=lp_post.timestamp+timedelta(seconds=prediction_horizons[horizon_key]))
    #     if i ==0:
    #         lp_prediction_tracks[horizon_key]=Track()
    #     lp_prediction_tracks[horizon_key].append(lp_forecasted_prediction)
                    
    if i==0:
        prior=gp_prior
    else:
        prior = GP_track[-1]
    gp_prediction = gp_predictor.predict(prior, timestamp=meas.timestamp)
    gp_hypothesis = SingleHypothesis(gp_prediction, meas)                    
    gp_post = gp_updater.update(gp_hypothesis)
    GP_track.append(gp_post)
    # for horizon_key in prediction_horizons:
    #     gp_forecasted_prediction = gp_predictor.predict(gp_post, timestamp=gp_post.timestamp+timedelta(seconds=prediction_horizons[horizon_key]))
    #     if i ==0:
    #         gp_prediction_tracks[horizon_key]=Track()
    #     gp_prediction_tracks[horizon_key].append(gp_forecasted_prediction)
    if (i+1)//100 == (i+1)/100:
        print(f"measurement {i+1} of {len(measurements)}")


In [ ]:
## Whether to calculate the smoothed trajectories and how to plot them
plot_smooth=False
uncertainty=True
particle=False
plot_particle_paths=False
i=0

In [ ]:
## Plotting

file_path = Path(folder_path + rf"\TrackingPlot_RW_mu_varied.html")
file_path.parent.mkdir(parents=True, exist_ok=True)
particle_plotter_dict[label]= Plotterly(autosize=False, width=1200,height=800, dimension=Dimension.ONE, axis_labels=[label])
if i==0:
    particle_plotter_dict[label].plot_ground_truths(observed_mid, [i], truths_label="Price Observations")
    particle_plotter_dict[label].plot_ground_truths(observed_bid, [i], truths_label="Bid")
    particle_plotter_dict[label].plot_ground_truths(observed_ask, [i], truths_label="Ask")

particle_plotter_dict[label].plot_tracks(LP_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Levy Filtered",line=dict(width=1))
particle_plotter_dict[label].plot_tracks(GP_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Gaussian Filtered",line=dict(width=1))

## currently pointless because model is still zero mean so prediction capabilities very limited.
# for horizon_key in prediction_horizons:
#     particle_plotter_dict[label].plot_tracks(lp_prediction_tracks[horizon_key], [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label=f"Levy prediction, lag = {horizon_key}",line=dict(width=1))
#     particle_plotter_dict[label].plot_tracks(gp_prediction_tracks[horizon_key], [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label=f"Gauss prediction, lag = {horizon_key}",line=dict(width=1))

if plot_smooth is True:
    from stonesoup.smoother.particle import CarterKohnSmoother, MarginalisedKalmanSmoother, ParticleSmoother
    particlesmoother=ParticleSmoother()
    culled_track=particlesmoother.particle_paths(track=LP_track)

    RTSsmoother=MarginalisedKalmanSmoother()
    RTS_track=RTSsmoother.smooth(track=LP_track)

    CKsmoother=CarterKohnSmoother()
    CK_track=CKsmoother.smooth(track=LP_track)
    particle_plotter_dict[label].plot_tracks(culled_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="culled",line=dict(width=1))
    particle_plotter_dict[label].plot_tracks(RTS_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="RTS",line=dict(width=1))
    particle_plotter_dict[label].plot_tracks(CK_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="CK",line=dict(width=1))

particle_plotter_dict[label].fig.update_layout( 
    plot_bgcolor="white",  # Set background color to white
    xaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text="Time", font=dict(size=20)),  # Add large label
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text=label, font=dict(size=20)),  # Add large label
    ),
    legend=dict(
        font=dict(size=15),       # Make the legend font larger
        # orientation='v',
        # xanchor="auto",         # Center the legend
        # yanchor="auto",           # Align the legend to the bottom of the plot
        bordercolor="Black",
        borderwidth=3,
        # y=+0.45,                   # Position it above the graph
        # x=0.6                    # Center it horizontally
    ),
)
particle_plotter_dict[label].fig.write_html(str(file_path))
# particle_plotter_dict[label].fig.show()

In [ ]:
## Generate OSPA plots to see how well each model has tracked the path
tracking_filters = ["Gaussian_filtered", 
                    "Levy_filtered"]

if plot_smooth is True:
    tracking_filters.append(["Levy_RTS", 
                            "Levy_CK"])

from stonesoup.metricgenerator.ospametric import OSPAMetric

ospa_generators = [OSPAMetric(c=40, p=1,
                            generator_name=f'{tracking_filter} OSPA metrics',
                            tracks_key=f'tracks_{tracking_filter}',
                            truths_key='truths'
                            )
                for tracking_filter in tracking_filters]

from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean

siap_generators = [SIAPMetrics(position_measure=Euclidean((0, 2)),
                            velocity_measure=Euclidean((1, 3)),
                            generator_name=f'{tracking_filter} SIAP metrics',
                            tracks_key=f'tracks_{tracking_filter}',
                            truths_key='truths'
                            )
                for tracking_filter in tracking_filters]

from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric

uncertainty_generators = [
    SumofCovarianceNormsMetric(generator_name=f'{tracking_filter} OSPA metrics',
                            tracks_key=f'tracks_{tracking_filter}')
    for tracking_filter in tracking_filters]

from stonesoup.dataassociator.tracktotrack import TrackToTruth
from stonesoup.metricgenerator.manager import MultiManager

associator = TrackToTruth(association_threshold=30)

generators = ospa_generators #+ siap_generators + uncertainty_generators
metric_manager = MultiManager(generators, associator=associator)

metric_manager.add_data({'truths': [observed_mid],
                        'tracks_Gaussian_filtered': [GP_track],
                        'tracks_Levy_filtered': [LP_track]})

if plot_smooth is True:
    metric_manager.add_data({
                        'tracks_Levy_RTS': [RTS_track],
                        'tracks_Levy_CK': [CK_track]
                        })
    
metrics = metric_manager.generate_metrics()

from stonesoup.plotter import MetricPlotter

# sum up distance error from ground truth over all timestamps
for tracking_filter in tracking_filters:
    total = sum([metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value[i].value
                for i in range(0, len(metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value))])
    print(f'OSPA total value for {tracking_filter} is {total:.3f}')

fig1 = MetricPlotter()
fig1.plot_metrics(metrics, metric_names=['OSPA distances'])